<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/03_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 103.1 MB/s eta 0:00:00


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 575 (delta 73), reused 17 (delta 7), pack-reused 450 (from 2)
Receiving objects: 100% (575/575), 563.87 KiB | 12.00 MiB/s, done.
Resolving deltas: 100% (348/348), done.


In [8]:
%cd /content/ML-Tech

/content/ML-Tech


In [9]:
!git pull origin main

From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [10]:
import faiss
index = faiss.read_index("data/processed/passport_index.faiss")

In [11]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "service":chunks[idx].get("service",""),
            "section": chunks[idx]["section"],
            "language":chunks[idx].get("language",""),
            "text": chunks[idx]["text"],
            "url": chunks[idx]["url"]
        })

    return results

In [18]:
results = retrieve(
    "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟"
)

for r in results:
    print(r["score"], r["service"], r["section"])

0.8270811438560486 "تجديد رخصة سوق منتهية الصلاحة" المستندات المطلوبة
0.8176745772361755 "تجديد رخصة سوق منتهية الصلاحة" الوصف
0.8095848560333252 "تجديد رخصة قيادة ضائعة" المستندات المطلوبة


In [20]:
results = retrieve(
    "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟",
    k=3
)
for result in results:
    print(result["section"], result["score"])

المستندات المطلوبة 0.8270811438560486
الوصف 0.8176745772361755
المستندات المطلوبة 0.8095848560333252


In [21]:
!pip install -q transformers accelerate

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [23]:
question = "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟"

results = retrieve(question, k=3)

In [24]:
context = "\n\n".join(
    f"Section: {result['section']}\n{result['text']}"
    for result in results
)

print(context)

Section: المستندات المطلوبة
2.1
رخصة السوق المنتهية الصالحية

2.2
سجل عدلي لا يعود تاريخه لأكثر من ثلاثة اشهر

2.3
بطاقة أو إفادة فئة الدم

2.4
شهادة طبية صادرة من نقابة الاطباء، وعليها الرسم الشمسي لصاحب العلاقة (لا يعود تاريخها لأكثر من ثلاثة اشهر)

2.5
صورة عن بطاقة الهوية أو إخراح قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين ابراز اقامة صالحة.

2.6
صورة شمسية مصدقة من المختار (عدد2)

3

Section: الوصف
(يستوجب حضور صاحب العلاقة شخصيا)
لا يتطلب موعد
 عند انتهاء تاريخ صلاحية رخصة السوق(خصوصية أوعمومية) يتوجب على المواطن تجديدها في المركز الذي صدرت عنه.

2

Section: المستندات المطلوبة
:

2.1
رخص السوق المفقودة الغير منتهية الصلاحية:

2.1.1
صورة طبق الاصل للمحضر المنظم لدى قوى الامن الداخلي.

2.1.2
صورة عن بطاقة الهوية أو إخراح قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين ابراز اقامة صالحة.

2.1.3
صورة شمسية مصدقة من المختار (عدد2)

2.2
رخصة السوق المفقودة منتهية الصلاحية (بالإضافة إلى المستندات أعلاه):

2.2.1
شهادة طبية صادرة من نقابة الاطباء، وعليها الرسم الشمسي لصاحب العلاق

In [28]:
def ask(question, k=3):

    # 1. Retrieve relevant chunks
    results = retrieve(question, k=k)

    # 2. Build context for the LLM
    context = "\n\n".join(
    f"""
Service: {result.get('service', '')}
Section: {result['section']}
Language: {result.get('language', '')}
Content:
{result['text']}
"""
    for result in results
)

    # 3. Create the prompt
    messages = [
       {
    "role": "system",
    "content": """
You are an assistant for Lebanese government procedures.

Answer the user's question using ONLY the information contained in the
provided official government context.

IMPORTANT:
- The context may be in Arabic, English, or French.
- The user's question may be in Arabic, English, or French.
- Answer in the same language as the user's question.
- Carefully read information written in Arabic.
- Use relevant information even if the wording of the question is different
  from the wording in the context.
- Do not require an exact phrase match between the question and the context.
- The retrieved context may contain irrelevant passages. Ignore those passages.
- If relevant information is present in any retrieved passage, use it to answer.
- Include all relevant documents, requirements, fees, and conditions found
  in the context.
- Do not invent information that is not present in the context.
- Only say that the information could not be found if NONE of the retrieved
  context contains information that answers the question.

Answer clearly and concisely.
"""
        },
        {
            "role": "user",
            "content": f"""
Official context:

{context}

Question:
{question}
"""
        }
    ]

    # 4. Convert messages into Qwen's chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    # 5. Generate answer
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # 6. Collect unique sources
    sources = []

    for result in results:
        url = result["url"]

        if url and url not in sources:
            sources.append(url)

    return {
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": results
    }

In [29]:
response = ask("ما هي المستندات لتجديد دفتر القيادة الخصوصي؟")

In [30]:
print(response["answer"])

print("\nSources:")
for source in response["sources"]:
    print(source)

لتجديد دفتر القيادة الخصوصي، المستندات المطلوبة هي:

2.1
رخصة السوق المفقودة الغير منتهية الصلاحية:
   - صورة طبق الأصل للمحضر المنظم لدى قوى الأمن الداخلي.
   - صورة عن بطاقة الهوية أو إخراج قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين إبراز إقامة صالحة.
   - صورة شمسية مصدقة من المختار (عدد 2).

2.2
رخصة السوق المفقودة منتهية الصلاحية (بالإضافة إلى المستندات أعلاه):
   - شهادة طبية صادرة من نقابة الأطباء، وعليها الرسم الشمسي لصاحب العلاقة (لا يعود تاريخها لأكثر من ثلاثة أشهر).
   - سجل عدلي لا يعود تاريخه لأكثر من ثلاثة أشهر.
   - بطاقة أو إفادة فئة الدم.

Sources:
https://tmo.gov.lb/web/panel/info/service-types/1
